# CCIS 675: Semantic Cybersecurity Log Analysis

Use verified **BOTS V3 event exports**, not native `.tsidx` or `journal.gz` files. No BOTS V3 experiment has yet been performed by this notebook. Semantic matches are candidates for investigation, not confirmed attacks. Select a GPU runtime optionally; CPU works too.

These source files must be pushed to GitHub before cloning, or upload and extract this working repository into `/content/semantic-log-analysis`.

In [ ]:
from pathlib import Path
import subprocess
ROOT = Path('/content/semantic-log-analysis')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/cbwash4/CCIS675-Assignment5-Semantic-Log-Analysis.git', str(ROOT)], check=True)
assert (ROOT / 'requirements.txt').exists(), 'Use the implemented version of this repository (push it or upload a ZIP).'
%cd /content/semantic-log-analysis
%pip install -r requirements.txt


## Mount Drive and choose a verified export
Change EVENT_FILE to your JSON, JSONL, or CSV export. Verify the export against Splunk before running. Keep real and synthetic databases separate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/botsv3/model_cache'
EVENT_FILE = Path('/content/drive/MyDrive/botsv3/exports/events.jsonl')
DB = Path('/content/drive/MyDrive/botsv3/forensic_db')
OUTPUT = Path('/content/drive/MyDrive/botsv3/results/investigations')
assert EVENT_FILE.is_file(), f'Provide a verified real event export: {EVENT_FILE}'
assert EVENT_FILE.suffix.lower() in {'.json', '.jsonl', '.csv'}, 'Native Splunk buckets are not event exports'


## Verify the software using labeled synthetic inputs
These checks do not validate BOTS V3 coverage or detection accuracy. The integration test downloads the actual MiniLM model.

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pytest', '-m', 'not integration', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-m', 'integration', '-q'], check=True)


## Inspect normalization and provenance
Record your Splunk export query, verified count, and time bounds in the report. Default selection covers EventIDs 1, 3, 11; add relevant types explicitly when needed.

In [ ]:
import hashlib
from semantic_logs.preprocessing import load_events
with EVENT_FILE.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
print({'file': str(EVENT_FILE), 'sha256': digest, 'bytes': EVENT_FILE.stat().st_size})
frame, counts = load_events([EVENT_FILE])
print(counts)
assert len(frame), 'No eligible events; inspect your export and EventID field extraction'
print(frame[['EventID', 'UtcTime', 'text_payload']].head().to_string(index=False))
del frame  # CLI reloads the export; release this preview's memory


## Generate embeddings and index
Each batch uses 384-dimensional MiniLM vectors. Device selection is automatic. Reuse the same DB only for intentional upserts; use a fresh directory for a clean experiment.

In [ ]:
import time
start = time.perf_counter()
subprocess.run([sys.executable, '-m', 'semantic_logs', 'index', str(EVENT_FILE),
                '--db', str(DB), '--batch-size', '64'], check=True)
print({'index_elapsed_seconds': time.perf_counter() - start})


## Run all ten exact assignment queries and export
Reports include processed/indexed counts from indexing, query/result counts, distances, and original records. Review events in context before drawing conclusions.

In [ ]:
start = time.perf_counter()
subprocess.run([sys.executable, '-m', 'semantic_logs', 'investigate', '--db', str(DB),
                '--queries', str(ROOT / 'config/queries.json'), '--output', str(OUTPUT)], check=True)
print({'ten_queries_elapsed_seconds': time.perf_counter() - start})
import json
print(json.loads(Path(str(OUTPUT) + '.summary.json').read_text()))
import pandas as pd
display(pd.read_csv(str(OUTPUT) + '.csv'))


## Optional filtered investigation
Times are inclusive UTC bounds. Missing/invalid times do not match. Adjust dates to the verified dataset time range.

In [ ]:
# Uncomment after selecting actual verified time bounds:
# subprocess.run([sys.executable, '-m', 'semantic_logs', 'query',
#                 'Unexpected outbound network connections', '--db', str(DB),
#                 '--event-ids', '3', '--after', '2018-08-01T00:00:00Z',
#                 '--before', '2018-09-01T00:00:00Z',
#                 '--output', str(OUTPUT.parent / 'network')], check=True)


## Report measurements honestly
Use the exported artifacts to fill `docs/methodology.md`. Record actual counts, hardware/device, versions, model revision, and measured timings. Precision/recall require independently reviewed relevance labels. Do not turn similarity scores into attack detections. Drive stores exports and database files; mounting and processes must be restarted in future sessions. Pandas loads each export in memory: split large exports into manageable shards. Do not share raw logs or results without authorization.